# FIR Document Preprocessing & AI Extraction Pipeline

A modular, step-by-step OpenCV document preprocessing and AI extraction preparation pipeline:
- **Step 1**: High-DPI Page Rendering & De-blurring (300 DPI)
- **Step 2**: Skew Detection & Deskewing (Projection Profile)
- **Step 3**: Dark Scanner Border Detection & Cropping to Paper Area (~20px margin)
- **Step 4**: Consistent White Background Normalization (Retinex Illumination Division & Color Preservation)
- **Step 5**: Form Header Identification (IIF 1 to IIF 5) & Content Filtering (Top 25% Crop)
- **Final**: Original vs Processed View & AI Extraction Payload Preparation


In [ ]:
import os
import re
import cv2
import numpy as np
import pypdfium2 as pdfium
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

# Pipeline Configuration
sample_pdf = "sample-docs/FIR-22-2010.pdf"
PAGES_TO_PROCESS = 5
TARGET_DPI = 300

print(f"Document: {sample_pdf} | Processing first {PAGES_TO_PROCESS} pages at {TARGET_DPI} DPI")


## Step 1: Render Pages at 300 DPI (with De-blurring)


In [ ]:
def deblur_unsharp_mask(image: np.ndarray, strength: float = 1.5) -> np.ndarray:
    """Sharpen text and handwriting strokes with Gaussian unsharp masking."""
    gaussian = cv2.GaussianBlur(image, (0, 0), sigmaX=2.0)
    return cv2.addWeighted(image, strength, gaussian, 1.0 - strength, 0)


def render_pdf_page(pdf_path: str, page_index: int = 0, dpi: int = TARGET_DPI) -> np.ndarray:
    """Render a single PDF page at target DPI with unsharp de-blurring."""
    doc = pdfium.PdfDocument(pdf_path)
    page = doc[page_index]
    w_pt, h_pt = page.get_size()
    scale = int(8.27 * dpi) / w_pt if w_pt > 1000 else dpi / 72.0
    bitmap = page.render(scale=scale)
    img = cv2.cvtColor(np.array(bitmap.to_pil()), cv2.COLOR_RGB2BGR)
    return deblur_unsharp_mask(img, strength=1.5)


# Render the first N pages
doc = pdfium.PdfDocument(sample_pdf)
num_pages = min(PAGES_TO_PROCESS, len(doc))
raw_pages = [render_pdf_page(sample_pdf, i, TARGET_DPI) for i in range(num_pages)]
print(f"Step 1 Complete: Rendered {len(raw_pages)} pages at {TARGET_DPI} DPI.")


## Step 2: Skew Detection & Deskewing (Projection Profile)


In [ ]:
def compute_skew_angle_projection(image: np.ndarray, max_angle: float = 10.0) -> float:
    """Detect document skew using horizontal projection profile variance."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image.copy()
    h, w = gray.shape[:2]
    small = cv2.resize(gray, (1000, int(h * (1000.0 / w))), interpolation=cv2.INTER_AREA)
    thresh = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    sh, sw = thresh.shape
    bx, by = int(sw * 0.02), int(sh * 0.02)
    clean_thresh = thresh[by:sh - by, bx:sw - bx]
    center = (clean_thresh.shape[1] // 2, clean_thresh.shape[0] // 2)
    
    coarse_angles = np.arange(-max_angle, max_angle + 0.5, 1.0)
    best_score, best_coarse = -1.0, 0.0
    for a in coarse_angles:
        M = cv2.getRotationMatrix2D(center, a, 1.0)
        rot = cv2.warpAffine(clean_thresh, M, (clean_thresh.shape[1], clean_thresh.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        p = np.sum(rot, axis=1, dtype=np.float64)
        score = np.sum((p[1:] - p[:-1]) ** 2)
        if score > best_score:
            best_score, best_coarse = score, a
            
    fine_angles = np.arange(best_coarse - 1.0, best_coarse + 1.05, 0.1)
    best_angle = best_coarse
    for a in fine_angles:
        M = cv2.getRotationMatrix2D(center, a, 1.0)
        rot = cv2.warpAffine(clean_thresh, M, (clean_thresh.shape[1], clean_thresh.shape[0]), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        p = np.sum(rot, axis=1, dtype=np.float64)
        score = np.sum((p[1:] - p[:-1]) ** 2)
        if score > best_score:
            best_score, best_angle = score, a
    return float(best_angle)


def deskew_image(image: np.ndarray) -> tuple:
    """Rotate image back to 0° using detected skew angle with clean white padding."""
    angle = compute_skew_angle_projection(image)
    if abs(angle) >= 0.1:
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        deskewed = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    else:
        deskewed = image.copy()
    return deskewed, angle


# Deskew all pages
deskewed_pages = [deskew_image(p)[0] for p in raw_pages]
print(f"Step 2 Complete: Deskewed {len(deskewed_pages)} pages to 0.00°.")


## Step 3: Scanner Border Detection & Cropping to Paper Area (~20px Margin)


In [ ]:
def crop_to_paper_area(
    image: np.ndarray,
    dark_threshold: int = 140,
    coverage_threshold: float = 0.35,
    safety_margin: int = 20
) -> tuple:
    """
    Detect black/grey scanner margins and gutter shadows, then crop to paper area with ~20px margin.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    h, w = gray.shape[:2]
    
    # 1. Left scanner margin / spine gutter shadow (search up to 20% width)
    max_left = int(w * 0.20)
    slice_l = gray[:, :max_left]
    col_means_l = slice_l.mean(axis=0)
    col_dark_l = (slice_l < dark_threshold).mean(axis=0)
    dark_l = np.where((col_means_l < dark_threshold) | (col_dark_l > coverage_threshold))[0]
    left = dark_l[-1] + 1 if len(dark_l) > 0 else 0
    while left < max_left and col_means_l[min(left, max_left - 1)] < 190:
        left += 1
        
    # 2. Right scanner margin (search right 12% width)
    min_right = int(w * 0.88)
    slice_r = gray[:, min_right:]
    col_means_r = slice_r.mean(axis=0)
    col_dark_r = (slice_r < dark_threshold).mean(axis=0)
    dark_r = np.where((col_means_r < dark_threshold) | (col_dark_r > coverage_threshold))[0]
    right = min_right + dark_r[0] if len(dark_r) > 0 else w
    while right > min_right and gray[:, min(w - 1, right - 1)].mean() < 190:
        right -= 1
        
    # 3. Top scanner margin (search top 12% height)
    max_top = int(h * 0.12)
    slice_t = gray[:max_top, :]
    row_means_t = slice_t.mean(axis=1)
    row_dark_t = (slice_t < dark_threshold).mean(axis=1)
    dark_t = np.where((row_means_t < dark_threshold) | (row_dark_t > coverage_threshold))[0]
    top = dark_t[-1] + 1 if len(dark_t) > 0 else 0
    while top < max_top and row_means_t[min(top, max_top - 1)] < 190:
        top += 1
        
    # 4. Bottom scanner margin (search bottom 12% height)
    min_bot = int(h * 0.88)
    slice_b = gray[min_bot:, :]
    row_means_b = slice_b.mean(axis=1)
    row_dark_b = (slice_b < dark_threshold).mean(axis=1)
    dark_b = np.where((row_means_b < dark_threshold) | (row_dark_b > coverage_threshold))[0]
    bottom = min_bot + dark_b[0] if len(dark_b) > 0 else h
    while bottom > min_bot and gray[min(h - 1, bottom - 1), :].mean() < 190:
        bottom -= 1
        
    # Apply ~20px safety margin so no text near the edge is clipped
    crop_left = max(0, left - safety_margin) if left > safety_margin else 0
    crop_right = min(w, right + safety_margin) if (w - right) > safety_margin else w
    crop_top = max(0, top - safety_margin) if top > safety_margin else 0
    crop_bottom = min(h, bottom + safety_margin) if (h - bottom) > safety_margin else h
    
    cropped_paper = image[crop_top:crop_bottom, crop_left:crop_right]
    return cropped_paper, (crop_left, crop_top, crop_right - crop_left, crop_bottom - crop_top)


# Crop all pages to clean paper area
paper_pages = [crop_to_paper_area(p, safety_margin=20)[0] for p in deskewed_pages]
print(f"Step 3 Complete: Cropped {len(paper_pages)} pages to paper area (20px margin).")


## Step 4: Consistent White Background Normalization (Retinex Illumination Division)

Achieve a uniform, consistent pure white (`#FFFFFF`) background across all pages regardless of original paper age (yellowed, gray, carbon duplicate) **without losing any content**:
- **Illumination Surface Modeling**: Isolates ambient paper lighting gradients via large-kernel morphological filtering without touching text or strokes.
- **Retinex Division ($R = Image / Illumination$)**: Mathematically shifts varying paper tones to clean white while preserving ink color saturation (blue ballpoint signatures, red/purple seals, black printed text).
- **Soft-Knee Highlight Snapping**: Smoothly maps near-white paper grain (luminosity >= 228) to absolute `#FFFFFF`, ensuring 100% background consistency across all pages.
- **Fine Diacritic Protection**: Preserves light pencil notes, stamps, and delicate Hindi *matras* with zero clipping or erosion.


In [ ]:
def normalize_background_color(
    image: np.ndarray,
    kernel_size: int = 61,
    snap_threshold: int = 228
) -> np.ndarray:
    """
    Normalize document background to a consistent pure white (#FFFFFF) across all pages
    via Retinex Illumination Division, preserving full color, faint stamps, and Hindi matras.
    """
    img_float = image.astype(np.float32)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 1. Estimate paper illumination surface (larger than text stroke width)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    bg_illum = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
    bg_illum = cv2.GaussianBlur(bg_illum, (31, 31), 0).astype(np.float32)
    
    # 2. Multi-channel broadcast and Illumination Division: Reflectance = Image / Illumination
    bg_3ch = np.stack([bg_illum, bg_illum, bg_illum], axis=-1)
    normalized = np.clip((img_float / (bg_3ch + 1e-5)) * 255.0, 0, 255.0)
    
    # 3. Soft-knee highlight snapping to guarantee uniform pure white (#FFFFFF) background
    norm_gray = cv2.cvtColor(normalized.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    mask = norm_gray >= snap_threshold
    
    # Smooth non-linear ramp from snap_threshold to 255.0 (no abrupt edges)
    weight = (norm_gray[mask].astype(np.float32) - snap_threshold) / (255.0 - snap_threshold)
    weight_3ch = np.stack([weight, weight, weight], axis=-1)
    
    snapped = normalized.copy()
    snapped[mask] = snapped[mask] * (1.0 - weight_3ch) + 255.0 * weight_3ch
    
    return np.clip(snapped, 0, 255).astype(np.uint8)


# Apply consistent white background normalization across all pages
processed_pages = [normalize_background_color(p) for p in paper_pages]
print(f"Step 4 Complete: Consistent white background normalized for {len(processed_pages)} pages.")


## Step 5: Form Header Identification (IIF 1 to IIF 5) & Content Filtering (Top 25% Crop)

Detect the standardized printed form header by cropping the top **25% region** of each page:
- **FIR FORM - IIF 1** (First Information Report / प्रथम सूचना रिपोर्ट, u/s 154 Cr.P.C.)
- **Crime Details FORM - IIF 2** (Scene of Crime / Physical Evidence)
- **Arrrest Details FORM - IIF 3** (Arrest / Court Surrender Form)
- **Property Seizure Form - IIF 4** (Property Search & Seizure Form, u/s 51, 102, 165 Cr.P.C.)
- **Final Report - IIF 5** (Final Form / Report / Charge Sheet / आरोप पत्र / चालान, u/s 173 Cr.P.C.)

Automatically detects and flags **sketch maps**, **blank pages**, and **signature-only pages** to mark them as **`skip text extraction`** so no AI tokens or resources are wasted on non-text pages.


In [ ]:
import os
import re
import cv2
import numpy as np

# Ensure reference template banners exist (bootstrapped from sample PDFs if missing)
os.makedirs("templates", exist_ok=True)
if not os.path.exists("templates/iif1_fir.png") and os.path.exists("sample-docs/FIR-22-2010.pdf"):
    _doc_ref = pdfium.PdfDocument("sample-docs/FIR-22-2010.pdf")
    _p1 = _doc_ref[0]
    _s1 = int(8.27 * 300) / _p1.get_size()[0] if _p1.get_size()[0] > 1000 else 300 / 72.0
    _img1 = cv2.cvtColor(np.array(_p1.render(scale=_s1).to_pil()), cv2.COLOR_RGB2GRAY)
    cv2.imwrite("templates/iif1_fir.png", _img1[:int(_img1.shape[0]*0.22), :][250:500, 400:2000][40:110, 130:980])
    
    _p5 = _doc_ref[4]
    _s5 = int(8.27 * 300) / _p5.get_size()[0] if _p5.get_size()[0] > 1000 else 300 / 72.0
    _img5 = cv2.cvtColor(np.array(_p5.render(scale=_s5).to_pil()), cv2.COLOR_RGB2GRAY)
    cv2.imwrite("templates/iif2_crime.png", _img5[:int(_img5.shape[0]*0.22), :][250:500, 400:2000][50:115, 170:780])
    
    _p9 = _doc_ref[8]
    _s9 = int(8.27 * 300) / _p9.get_size()[0] if _p9.get_size()[0] > 1000 else 300 / 72.0
    _img9 = cv2.cvtColor(np.array(_p9.render(scale=_s9).to_pil()), cv2.COLOR_RGB2GRAY)
    cv2.imwrite("templates/iif5_final.png", _img9[:int(_img9.shape[0]*0.22), :][250:500, 400:2000][45:110, 170:620])

if not os.path.exists("templates/iif4_seizure.png") and os.path.exists("sample-docs/FIR-2-2009.pdf"):
    _doc_ref2 = pdfium.PdfDocument("sample-docs/FIR-2-2009.pdf")
    _p_sz = _doc_ref2[8]
    _s_sz = int(8.27 * 300) / _p_sz.get_size()[0] if _p_sz.get_size()[0] > 1000 else 300 / 72.0
    _img_sz = cv2.cvtColor(np.array(_p_sz.render(scale=_s_sz).to_pil()), cv2.COLOR_RGB2GRAY)
    cv2.imwrite("templates/iif4_seizure.png", _img_sz[:int(_img_sz.shape[0]*0.22), :][250:500, 400:2000][100:160, 270:1230])

# Standardized Indian Police Integrated Investigation Forms (IIF) Taxonomy
FORM_TAXONOMY = {
    "IIF-1": {
        "label": "FIR FORM - IIF 1",
        "template_file": "templates/iif1_fir.png",
        "threshold": 0.70,
        "ai_prompt": (
            "Extract all fields from this FIR (First Information Report - IIF 1) in JSON format: "
            "District, Police Station, Year, FIR No, Date, Acts & Sections, Place of Occurrence, "
            "Complainant / Informant details, and Details of Known / Suspected / Unknown Accused."
        ),
        "patterns": [
            r"first\s*information\s*report",
            r"\bfir\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*1\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*i\b",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*1",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*[I।I]",
            r"प्रथम\s*सूचना\s*रिपोर्ट",
            r"154\s*(?:\(1\))?\s*cr\.?\s*p\.?\s*c",
        ],
    },
    "IIF-2": {
        "label": "Crime Details FORM - IIF 2",
        "template_file": "templates/iif2_crime.png",
        "threshold": 0.70,
        "ai_prompt": (
            "Extract all fields from this Crime Details Form (IIF 2) in JSON format: "
            "District, Police Station, FIR No, Acts and Sections, Place of occurrence description, "
            "Witness names and addresses, Date and time of visit, and Physical evidence recovered."
        ),
        "patterns": [
            r"crime\s*details(?:\s*form)?",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*2\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*ii\b",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*2",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*ii",
            r"अपराध\s*विवरण\s*फार्म",
            r"scene\s*of\s*crime",
            r"घटना\s*स्थल",
        ],
    },
    "IIF-3": {
        "label": "Arrrest Details FORM - IIF 3",
        "template_file": None,
        "threshold": 0.70,
        "ai_prompt": (
            "Extract all fields from this Arrest / Court Surrender Form (IIF 3) in JSON format: "
            "Arrestee Name, Father Name, Age, Address, Date & Time of Arrest, Arresting Officer, and Grounds of Arrest."
        ),
        "patterns": [
            r"arrest(?:(?:\s*details)?(?:\s*form)?|(?:\s*(?:\/|and)\s*court)?\s*surrender(?:\s*form)?)",
            r"arrest\s*memo",
            r"court\s*surrender",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*3\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*iii\b",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*3",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*iii",
            r"गिरफ्तारी|गिरफ़्तारी",
        ],
    },
    "IIF-4": {
        "label": "Property Seizure Form - IIF 4",
        "template_file": "templates/iif4_seizure.png",
        "threshold": 0.70,
        "ai_prompt": (
            "Extract all fields from this Property Search & Seizure Form (IIF 4) in JSON format: "
            "Seizure Memo details, Property recovered/seized, Location, Witnesses to search, and Officer details."
        ),
        "patterns": [
            r"property\s*(?:search\s*(?:&|and)\s*)?seizure(?:(?:\s*search)?\s*form)?",
            r"seizure\s*memo",
            r"recovery\s*memo",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*4\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*iv\b",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*4",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*iv",
            r"सम्पत्ति\s*की\s*तलाशी",
            r"जब्ती\s*फार्म|जब्ती\s*सूची",
            r"(?:51|102|165)\\s*cr\.?\s*p\.?\s*c",
        ],
    },
    "IIF-5": {
        "label": "Final Report - IIF 5",
        "template_file": "templates/iif5_final.png",
        "threshold": 0.75,
        "ai_prompt": (
            "Extract all fields from this Final Report / Charge Sheet (IIF 5) in JSON format: "
            "Court name, FIR No, Date, Acts/Sections, Accused persons charge-sheeted, Accused persons not charge-sheeted, and Witnesses."
        ),
        "patterns": [
            r"final\s*(?:form\s*(?:\/|and)\s*)?report",
            r"charge\s*sheet",
            r"closure\s*report",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*5\b",
            r"i[\.\s]*i[\.\s]*f[\.\s]*-?[\s]*v\b",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*5",
            r"एकीकृत\s*जाँच\s*फार्म[\s\-]*v",
            r"अंतिम\s*फार्म|अंतिम\s*रिपोर्ट",
            r"आरोप\s*पत्र|चालान",
            r"173\s*cr\.?\s*p\.?\s*c",
        ],
    },
}


def extract_header_roi(page_image: np.ndarray, header_ratio: float = 0.25) -> np.ndarray:
    """Extract top 25% region of the page to detect form header."""
    h = page_image.shape[0]
    return page_image[: int(h * header_ratio), :]


def detect_page_content_type(page_image: np.ndarray, header_text: str = "") -> dict:
    """
    Detect special page categories (blank pages, sketch maps, signature-only pages)
    and mark them to skip text extraction.
    """
    gray = cv2.cvtColor(page_image, cv2.COLOR_BGR2GRAY) if len(page_image.shape) == 3 else page_image
    h, w = gray.shape[:2]
    
    # 1. Header keyword check for sketch maps
    header_lower = header_text.lower()
    map_keywords = ["sketch", "map of the place", "नक्शा मौका", "रेखाचित्र", "site plan", "place of occurrence"]
    if any(k in header_lower for k in map_keywords):
        return {
            "page_type": "SKETCH_MAP",
            "skip_text_extraction": True,
            "skip_reason": "Sketch Map (Site Plan)",
        }
        
    # 2. Ink density analysis on page core (excluding outer margin noise)
    _, binary = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
    margin_y, margin_x = int(h * 0.05), int(w * 0.05)
    core = binary[margin_y:h - margin_y, margin_x:w - margin_x]
    ink_ratio = cv2.countNonZero(core) / float(core.size)
    core_mean = float(gray[margin_y:h - margin_y, margin_x:w - margin_x].mean())
    
    # A. Blank Page: virtually no ink
    if ink_ratio < 0.006 or core_mean > 251.0:
        return {
            "page_type": "BLANK_PAGE",
            "skip_text_extraction": True,
            "skip_reason": "Blank Page",
        }
        
    # B. Signature-Only Page: very low ink concentrated in only 1-2 small regions, >80% empty
    bands = np.array_split(core, 6, axis=0)
    band_ratios = [cv2.countNonZero(b) / float(b.size) for b in bands]
    empty_bands = sum(1 for r in band_ratios if r < 0.003)
    
    sig_keywords = ["signature of officer", "signature of investigating", "थानाधिकारी", "जाँच अधिकारी"]
    has_sig_keyword = any(k in header_lower for k in sig_keywords)
    
    if (ink_ratio < 0.018 and empty_bands >= 4) or (has_sig_keyword and ink_ratio < 0.025):
        return {
            "page_type": "SIGNATURE_ONLY",
            "skip_text_extraction": True,
            "skip_reason": "Signature-Only Page",
        }
        
    # C. Sketch Map via Contour Analysis: large continuous hand-drawn geometric boundaries
    mid_core = core[int(h * 0.20):int(h * 0.75), :]
    mid_ink = cv2.countNonZero(mid_core) / float(mid_core.size)
    contours, _ = cv2.findContours(mid_core, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large_contours = [c for c in contours if cv2.contourArea(c) > (h * w * 0.015)]
    
    if len(large_contours) >= 1 and mid_ink < 0.08:
        proj = np.sum(mid_core, axis=1)
        peaks = (proj > (proj.mean() + proj.std())).sum()
        if peaks < 15:
            return {
                "page_type": "SKETCH_MAP",
                "skip_text_extraction": True,
                "skip_reason": "Sketch Map (Site Plan)",
            }
            
    return {
        "page_type": "STANDARD_FORM",
        "skip_text_extraction": False,
        "skip_reason": None,
    }


def classify_text_indic(text: str) -> tuple:
    """Classify extracted text using Indic / bilingual Hindi-English regex patterns."""
    text_clean = text.lower().replace("\n", " ")
    for form_id, info in FORM_TAXONOMY.items():
        for pat in info["patterns"]:
            if re.search(pat, text_clean, re.IGNORECASE):
                return form_id, info["label"], pat
    return "UNKNOWN", "Unidentified Document / Continuation", None


def match_visual_header_banner(gray_header: np.ndarray) -> tuple:
    """Multi-scale template matcher against standardized printed IIF form header banners."""
    best_id, best_score = None, 0.0
    for fid, info in FORM_TAXONOMY.items():
        tmpl_path = info.get("template_file")
        if not tmpl_path or not os.path.exists(tmpl_path):
            continue
        tmpl = cv2.imread(tmpl_path, cv2.IMREAD_GRAYSCALE)
        if tmpl is None:
            continue
            
        thresh = info.get("threshold", 0.50)
        for s in [0.90, 0.95, 1.0, 1.05, 1.10]:
            th, tw = int(tmpl.shape[0] * s), int(tmpl.shape[1] * s)
            if th >= gray_header.shape[0] or tw >= gray_header.shape[1]:
                continue
            scaled = cv2.resize(tmpl, (tw, th))
            res = cv2.matchTemplate(gray_header, scaled, cv2.TM_CCOEFF_NORMED)
            val = float(res.max())
            if val > thresh and val > best_score:
                best_score = val
                best_id = fid
                
    return best_id, best_score


def identify_form_header(page_image: np.ndarray, page_index: int = 0, prev_form: str = None) -> dict:
    """
    Extract top 25% header and classify page type, form ID, and clean form label:
    - FIR FORM - IIF 1
    - Crime Details FORM - IIF 2
    - Arrrest Details FORM - IIF 3
    - Property Seizure Form - IIF 4
    - Final Report - IIF 5
    """
    header_roi = extract_header_roi(page_image, header_ratio=0.25)
    gray_roi = cv2.cvtColor(header_roi, cv2.COLOR_BGR2GRAY) if len(header_roi.shape) == 3 else header_roi
    ocr_text = ""
    
    # 1. Try PyTesseract with Indic (hin+eng) support if installed
    try:
        import pytesseract
        thresh_roi = cv2.threshold(gray_roi, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
        try:
            ocr_text = pytesseract.image_to_string(thresh_roi, lang='hin+eng', config='--psm 6')
        except Exception:
            ocr_text = pytesseract.image_to_string(thresh_roi, config='--psm 6')
    except Exception:
        pass
        
    # 2. Check special page categories (blank pages, sketch maps, signature-only)
    page_type_info = detect_page_content_type(page_image, header_text=ocr_text)
    
    # 3. Classify extracted text against IIF taxonomy
    form_id, form_label, matched_pattern = classify_text_indic(ocr_text)
    is_continuation = False
    
    # 4. Multi-scale visual banner matching on top 25% crop
    if form_id == "UNKNOWN":
        banner_id, banner_score = match_visual_header_banner(gray_roi)
        if banner_id:
            form_id = banner_id
            form_label = FORM_TAXONOMY[banner_id]["label"]
            
    # 5. Continuation tracking
    if form_id == "UNKNOWN":
        if prev_form:
            form_id = prev_form
            form_label = f"{FORM_TAXONOMY[prev_form]['label']} (Continuation)"
            is_continuation = True
        else:
            form_id = "IIF-1"
            form_label = FORM_TAXONOMY["IIF-1"]["label"]
            
    base_id = form_id if form_id in FORM_TAXONOMY else "IIF-1"
    ai_prompt = FORM_TAXONOMY[base_id]["ai_prompt"]
    
    return {
        "page": page_index + 1,
        "form_id": form_id,
        "form_label": form_label,
        "is_continuation": is_continuation,
        "page_type": page_type_info["page_type"],
        "skip_text_extraction": page_type_info["skip_text_extraction"],
        "skip_reason": page_type_info["skip_reason"],
        "ai_prompt": ai_prompt,
    }


# Run Step 5 Form Header Identification on all processed pages
page_analysis_results = []
current_active_form = None

print("=== DOCUMENT FORM IDENTIFICATION (Top 25% Page Crop) ===\n")
for idx, page_img in enumerate(processed_pages):
    result = identify_form_header(page_img, page_index=idx, prev_form=current_active_form)
    page_analysis_results.append(result)
    current_active_form = result["form_id"]
    
    # Print clean summary: Page Number | Form Label | Skip Status (if applicable)
    skip_str = f" | [SKIP TEXT EXTRACTION: {result['skip_reason']}]" if result["skip_text_extraction"] else ""
    print(f"Page {result['page']} | {result['form_label']}{skip_str}")


## Final Step: Original vs Processed View & AI Extraction Preparation

1. **Side-by-side visual comparison**: **Original Scan** vs **Processed View** with standard form header and skip status banner.
2. **AI Extraction Payloads**: Prepares structured payloads with tailored extraction prompts ready to be dispatched to a multimodal AI vision model (e.g. Gemini).


In [ ]:
# 1. Visual Comparison: Original Scan vs Processed View
for i in range(num_pages):
    fig, axes = plt.subplots(1, 2, figsize=(16, 12))
    res = page_analysis_results[i]
    
    # Original Page (as loaded from PDF)
    axes[0].imshow(cv2.cvtColor(raw_pages[i], cv2.COLOR_BGR2RGB), interpolation='none')
    axes[0].set_title(f"Page {i + 1}: Original Scan", fontsize=13)
    axes[0].axis('off')
    
    # Processed Page (Clean view with Page Number and Standard Form Label)
    axes[1].imshow(cv2.cvtColor(processed_pages[i], cv2.COLOR_BGR2RGB), interpolation='none')
    
    if res["skip_text_extraction"]:
        title_str = f"Page {i + 1} | {res['form_label']}\n*** SKIP TEXT EXTRACTION: {res['skip_reason']} ***"
        title_color = 'darkred'
    else:
        title_str = f"Page {i + 1} | {res['form_label']}"
        title_color = 'navy'
        
    axes[1].set_title(title_str, fontsize=13, fontweight='bold', color=title_color)
    axes[1].axis('off')
    
    # If page is marked to skip, draw a clear alert banner across the top
    if res["skip_text_extraction"]:
        axes[1].text(
            0.5, 0.96, f"SKIP TEXT EXTRACTION: {res['skip_reason'].upper()}",
            transform=axes[1].transAxes, fontsize=14, fontweight='bold',
            color='white', ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='red', alpha=0.85, edgecolor='darkred')
        )
    
    plt.tight_layout()
    plt.show()

# 2. Prepare Structured Payloads Ready to Send to Multimodal AI for Text Extraction
ai_extraction_payloads = []

print("\n=== PREPARED AI EXTRACTION PAYLOADS (READY TO SEND TO AI) ===\n")
for i, res in enumerate(page_analysis_results):
    if res["skip_text_extraction"]:
        payload = {
            "page": res["page"],
            "form": res["form_label"],
            "status": "SKIPPED",
            "action": "skip_text_extraction",
            "reason": res["skip_reason"],
            "image": None,
        }
        print(f"Page {res['page']} [{res['form_label']}]: SKIPPED ({res['skip_reason']}) - 0 AI tokens spent.")
    else:
        # Preprocessed RGB image ready for AI vision API
        page_rgb = cv2.cvtColor(processed_pages[i], cv2.COLOR_BGR2RGB)
        payload = {
            "page": res["page"],
            "form": res["form_label"],
            "status": "READY_FOR_AI_EXTRACTION",
            "action": "send_to_ai",
            "image_shape": page_rgb.shape,
            "ai_prompt": res["ai_prompt"],
        }
        print(f"Page {res['page']} [{res['form_label']}]: READY FOR AI EXTRACTION")
        print(f"       Prompt: {res['ai_prompt'][:95]}...\n")
        
    ai_extraction_payloads.append(payload)

print(f"Total Pages: {len(ai_extraction_payloads)} | Ready for AI: {sum(1 for p in ai_extraction_payloads if p['status'] == 'READY_FOR_AI_EXTRACTION')} | Skipped: {sum(1 for p in ai_extraction_payloads if p['status'] == 'SKIPPED')}")
